# 01_14_A — Exemplo de código com apoio de LLM local usando modelo deepseek-coder:6.7b e a placa GPU


> **Material de familiarização com o JupyterLab do SimServ.**  
> Estes notebooks foram pensados para ficar em uma pasta de somente leitura, por exemplo `/home/jovyan/recursos_e_exercicios`.  
> Para editar, testar livremente ou salvar resultados, faça uma cópia para sua área de trabalho, por exemplo `/home/jovyan/ambiente_laboratorio`.

Este notebook usa o serviço local **Ollama** do SimServ via API HTTP, sem depender de serviços externos.

## Objetivo

Mostrar um exemplo simples de como um modelo de linguagem local pode explicar um trecho de código Python.

O modelo configurado é:

```text
deepseek-coder:6.7b
```
Obs. - Este é um dos modelos instalados no SimServ. É um *modelo de linguagem* especializado em programação e desenvolvimento de software. Ele ocupa entre 4 GB e 8 GB de memória VRAM, dependendo do método de otimização (quantização). O SimServ dispõe de 24 GB de memória VRAM. É recomendado para: Autocompletar código em tempo real: sugestões inteligentes enquanto você digita em editores de texto como VS Code; Geração de blocos de código: criar funções, classes ou scripts completos a partir de um prompt (ex: "crie uma API em Python usando FastAPI"); Refatoração e Depuração: explicar erros, sugerir melhorias e encontrar bugs em trechos de código existentes e produzir documentação: gerar comentários e docstrings automaticamente.

O notebook foi ajustado para:

1. corrigir os erros de indentação do notebook original;
2. localizar o serviço Ollama em `http://ollama:11434` ou endereços locais equivalentes;
3. verificar se o modelo está disponível;
4. consultar o modelo com `stream=False`;
5. exibir informações de diagnóstico, incluindo o estado retornado por `/api/ps`;
6. continuar didaticamente mesmo quando o Ollama ou o modelo não estiverem disponíveis.

ADICIONAR E OTIMIZAR: Explicação sobre o Ollama:
O Ollama é o **servidor local de inferência de modelos LLM**. No SimServ, ele é usado como base para uma **camada de processamento linguístico-semântico**, responsável por análise de narrativas, extração estruturada, síntese textual, geração de especificações e apoio à geração de código.
No SimServ, o Ollama, adequadamente configurado, pode: gerar texto, estruturar conversas (chats), gerar especificações formais para notebooks e simulações em arquivos JSON, realizar buscas semânticas,listar modelos locais e seu estado, inspecionar modelos, fazer diagnóstico do ambiente. Neste notebook usamos algumas dessas potencialidades. 

In [ ]:
# Configuração inicial
from dotenv import load_dotenv
load_dotenv()
import os


## Modo de execução desta versão

**Versão A — GPU.**

Esta versão solicita ao Ollama que carregue o modelo com máximo offload para GPU usando a opção `num_gpu: 999`.

No SimServ com RTX 3090, o modelo `deepseek-coder:6.7b` deve caber em VRAM. Ainda assim, o notebook verifica `/api/ps` ao final, porque quem efetivamente executa o modelo é o serviço Ollama.

In [ ]:
from textwrap import dedent

codigo_exemplo = dedent("""
for i in range(5):
    print(i)
""").strip()

print("Código que será explicado:\n")
print(codigo_exemplo)

ADICIONAR E OTIMIZAR:  O código a seguir faz ... e aqui são definidas as opções do modelo (EXPLICAR QUAIS SÃO AS OPÇÕES DISPONÍVEIS). Comentar no código o que faz cada função def.

In [ ]:
import json
import os
import socket
import time
from urllib.request import Request, urlopen

MODELO_OLLAMA = os.getenv('OLLAMA_MODEL', 'ministral-3:3b')
MODO_EXECUCAO = "Ollama Cloud"

def detectar_ollama():
    base = os.getenv('OLLAMA_BASE_URL', 'https://api.ollama.com').rstrip('/')
    api_key = os.getenv('OLLAMA_API_KEY')
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
    if api_key:
        headers['Authorization'] = f'Bearer {api_key}'

    urls = [base]
    for fb in ['http://ollama:11434', 'http://127.0.0.1:11434', 'http://localhost:11434']:
        if fb not in urls:
            urls.append(fb)

    for url_base in urls:
        try:
            req = Request(url_base + '/api/tags', headers=headers)
            with urlopen(req, timeout=5) as resp:
                dados = json.loads(resp.read().decode('utf-8'))
            if 'models' in dados:
                return url_base, "disponivel"
        except Exception:
            pass
    return None, None


def consultar_ollama_cloud(prompt, modelo=MODELO_OLLAMA, base=None):
    if base is None:
        base = os.getenv('OLLAMA_BASE_URL', 'https://api.ollama.com').rstrip('/')
    api_key = os.getenv('OLLAMA_API_KEY')
    headers = {
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json',
    }
    if api_key:
        headers['Authorization'] = f'Bearer {api_key}'

    payload = {
        'model': modelo,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
        'options': {'temperature': 0.2, 'top_p': 0.9, 'num_predict': 300},
    }
    req = Request(
        base + '/api/chat',
        data=json.dumps(payload).encode('utf-8'),
        headers=headers,
        method='POST',
    )
    with urlopen(req, timeout=180) as resp:
        return json.loads(resp.read().decode('utf-8'))


print("=== Ambiente ===")
print("Host/container:", socket.gethostname())
print("Diretório atual:", os.getcwd())
print("Modo solicitado:", MODO_EXECUCAO)
print("Modelo solicitado:", MODELO_OLLAMA)


In [ ]:
base_ollama, versao_ollama = detectar_ollama()

if not base_ollama:
    print("Ollama não foi localizado (nem cloud, nem local).")
    print("Configure OLLAMA_API_KEY no .env para usar a Ollama Cloud.")
else:
    print("Ollama localizado em:", base_ollama)


Explicar o código da próxima célula: o que faz e qual resultado que produz

In [ ]:
prompt = f"""
Explique o código Python abaixo para um iniciante.

Requisitos:
- responda em português;
- seja claro, breve e didático;
- explique o papel de range(5), da variável i e de print(i);
- não gere código novo, apenas explique.

Código:
{codigo_exemplo}
""".strip()

print(prompt)

In [ ]:
if not base_ollama:
    resposta = None
    dados_resposta = None
else:
    print(f"Consultando {MODELO_OLLAMA} em {base_ollama}...")
    inicio = time.time()
    try:
        dados_resposta = consultar_ollama_cloud(prompt, modelo=MODELO_OLLAMA, base=base_ollama)
        resposta = dados_resposta.get('message', {}).get('content', '').strip()
        if not resposta:
            resposta = dados_resposta.get('response', '').strip()
        duracao = time.time() - inicio
        print(f"Consulta concluída em {duracao:.1f} s.")
    except Exception as erro:
        resposta = None
        dados_resposta = None
        print("A consulta ao Ollama falhou:")
        print(type(erro).__name__, erro)

if resposta:
    print("\nResposta da IA:")
    print(resposta)
else:
    print("\nA IA não respondeu neste momento.")
    print("\nExplicação alternativa:")
    print('O comando for repete uma ação.')
    print('Neste exemplo, a variável i assume os valores 0, 1, 2, 3 e 4.')
    print('A função print(i) mostra cada valor na tela.')


Explicar o que faz e que resultado produz o próximo código

In [ ]:
if base_ollama:
    print("=== Consulta concluída ===")
    print("Modelo utilizado:", MODELO_OLLAMA)
    print("Endpoint:", base_ollama)


Próximo código deve salvar o relatório do resultado na pasta /ambiente_laboratorio/analise_codigo_ollama_GPU/analise_codigo_GPU_DATA_HORA.md

## Sugestões de trabalho para o aluno

1. Modifique o código de exemplo para imprimir os números de 0 a 9.
2. Rode novamente as células.
3. Compare a explicação gerada pelo modelo com a explicação alternativa.
4. Observe o diagnóstico final para verificar se o modo esperado foi usado.